# 1\.1\.3 Load Bridges and Tunnels

This notebook loads and validates the MTA Bridges and Tunnels Hourly Crossings dataset for use in the NYC congestion pricing mobility analysis pipeline\. The dataset contains hourly traffic counts segmented by facility, travel direction, payment method, and vehicle classification across the MTA bridge and tunnel network\. The primary objectives of this notebook are to inspect the source data, validate field completeness and categorical coverage, identify any potential data quality issues, and stage the dataset into a standardized parquet format for downstream temporal and spatial standardization\. Bridge and tunnel crossings provide an important multimodal perspective on regional travel behavior and may help quantify how congestion pricing influences traffic flows across key vehicular entry and exit points serving New York City\.

In [1]:
# ---------------------------------------------------------------------
# 1.1.3 Load MTA Bridges and Tunnels Hourly Crossings
# ---------------------------------------------------------------------

from pathlib import Path

import pandas as pd

## 1\.1\.3\.1 Load Bridges & Tunnels data

In [2]:
# ---------------------------------------------------------------------
# 1.1.3.1 Load Bridges & Tunnels data
# ---------------------------------------------------------------------

SOURCE_DATA_DIR = Path("source_data")
PIPELINE_DATA_DIR = Path("pipeline_data")

BRIDGE_TUNNEL_RAW_PATH = (
    SOURCE_DATA_DIR /
    "MTA_Bridges_and_Tunnels_Hourly_Crossings__Beginning_2019_20260530.csv"
)

BRIDGE_TUNNEL_OUTPUT_DIR = (
    PIPELINE_DATA_DIR /
    "1.1.3.bridge_tunnel_hourly_crossings"
)

BRIDGE_TUNNEL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BRIDGE_TUNNEL_STAGED_PARQUET_PATH = (
    BRIDGE_TUNNEL_OUTPUT_DIR /
    "bridge_tunnel_hourly_crossings_staged.parquet"
)

In [3]:
# ---------------------------------------------------------------------
# Define Bridge & Tunnel source and output paths
# ---------------------------------------------------------------------
# The 1.1.3 output directory is the canonical staged location expected by
# downstream temporal-standardization notebooks. Keep this path aligned with
# the notebook number so a clean rerun writes Bridge/Tunnel outputs where
# later pipeline steps already look for them.

SOURCE_DATA_DIR = Path("source_data")
PIPELINE_DATA_DIR = Path("pipeline_data")

BRIDGE_TUNNEL_RAW_PATH = (
    SOURCE_DATA_DIR /
    "MTA_Bridges_and_Tunnels_Hourly_Crossings__Beginning_2019_20260530.csv"
)

BRIDGE_TUNNEL_OUTPUT_DIR = (
    PIPELINE_DATA_DIR /
    "1.1.3.bridge_tunnel_hourly_crossings"
)

BRIDGE_TUNNEL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BRIDGE_TUNNEL_STAGED_PARQUET_PATH = (
    BRIDGE_TUNNEL_OUTPUT_DIR /
    "bridge_tunnel_hourly_crossings_staged.parquet"
)

In [4]:
# ---------------------------------------------------------------------
# Confirm raw Bridge & Tunnel file exists
# ---------------------------------------------------------------------

if not BRIDGE_TUNNEL_RAW_PATH.exists():
    raise FileNotFoundError(
        f"Bridge & Tunnel raw file not found: {BRIDGE_TUNNEL_RAW_PATH}"
    )

print(f"Raw Bridge & Tunnel file found: {BRIDGE_TUNNEL_RAW_PATH}")
print(f"File size: {BRIDGE_TUNNEL_RAW_PATH.stat().st_size / (1024 ** 2):,.2f} MB")

Raw Bridge & Tunnel file found: source_data/MTA_Bridges_and_Tunnels_Hourly_Crossings__Beginning_2019_20260530.csv
File size: 837.95 MB


In [5]:
# ---------------------------------------------------------------------
# Load raw Bridge & Tunnel CSV
# ---------------------------------------------------------------------

bridge_tunnel_raw = pd.read_csv(
    BRIDGE_TUNNEL_RAW_PATH,
    low_memory=False
)

print("Raw Bridge & Tunnel shape:", bridge_tunnel_raw.shape)

bridge_tunnel_raw.head()

Raw Bridge & Tunnel shape: (5941375, 11)


,Transit Timestamp,Date,Hour,Facility ID,Facility,Direction,Payment Method,Vehicle Class,Vehicle Class Description,Vehicle Class Category,Traffic Count
0,01/01/2023 12:00:00 AM,01/01/2023,0,27,Queens Midtown Tunnel,Westbound to Manhattan,Tolls by Mail,9,motorcycle,Motorcycle,2
1,01/01/2023 12:00:00 AM,01/01/2023,0,28,Hugh L. Carey Tunnel,Northbound to Manhattan,Tolls by Mail,1,2-axle passenger car,Car,83
2,01/01/2023 12:00:00 AM,01/01/2023,0,29,Throgs Neck Bridge,Southbound to Queens,E-ZPass,6,3-axle truck,Truck,1
3,01/01/2023 12:00:00 AM,01/01/2023,0,28,Hugh L. Carey Tunnel,Northbound to Manhattan,E-ZPass,1,2-axle passenger car,Car,404
4,01/01/2023 12:00:00 AM,01/01/2023,0,30,Verrazzano - Narrows Bridge,Eastbound to Brooklyn,E-ZPass,8,5-axle truck,Truck,3


## 1\.1\.3\.2 Inspect bridges and tunnels data schema \+ rename columns

In [6]:
# ---------------------------------------------------------------------
# 1.1.3.2 Inspect bridges and tunnels data schema
# ---------------------------------------------------------------------

bridge_tunnel_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5941375 entries, 0 to 5941374
Data columns (total 11 columns):
 #   Column                     Dtype 
---  ------                     ----- 
 0   Transit Timestamp          object
 1   Date                       object
 2   Hour                       int64 
 3   Facility ID                int64 
 4   Facility                   object
 5   Direction                  object
 6   Payment Method             object
 7   Vehicle Class              int64 
 8   Vehicle Class Description  object
 9   Vehicle Class Category     object
 10  Traffic Count              int64 
dtypes: int64(4), object(7)
memory usage: 498.6+ MB


In [7]:
# ---------------------------------------------------------------------
# Preview columns
# ---------------------------------------------------------------------

bridge_tunnel_raw.columns.tolist()

['Transit Timestamp',
 'Date',
 'Hour',
 'Facility ID',
 'Facility',
 'Direction',
 'Payment Method',
 'Vehicle Class',
 'Vehicle Class Description',
 'Vehicle Class Category',
 'Traffic Count']

Data Dictionary

Transit Timestamp\. Timestamp associated with the bridge or tunnel crossing observation\. This field represents the primary temporal dimension of the dataset and will be used to derive dates, hours, days of week, temporal buckets, and congestion\-pricing periods\.
Date\. Calendar date associated with the crossing observation\. This field provides the date\-level temporal reference for aggregation and validation\.
Hour\. Hour of day associated with the crossing observation\. Crossings are rounded up to the nearest hour in the source data, so this field should be interpreted as an hourly reporting bucket rather than an exact crossing time\.
Facility ID\. Unique identifier for the bridge or tunnel facility associated with the crossing observation\. More reliable than facility names for joins, QA, and aggregation\.
Facility\. Human\-readable name of the bridge or tunnel facility associated with the crossing observation\.
Direction\. Travel direction associated with the crossing observation\. This field can be used to distinguish inbound and outbound traffic patterns where available\.
Payment Method\. Toll payment method associated with the crossing observation, such as E\-ZPass, Tolls by Mail, or other payment categories represented in the source data\.
Vehicle Class\. Vehicle class code associated with the crossing observation\. This field provides a compact vehicle classification value for grouping and QA\.
Vehicle Class Description\. Human\-readable description of the vehicle class associated with the crossing observation\.
Vehicle Class Category\. Higher\-level vehicle class grouping associated with the crossing observation\. This field can be used to aggregate detailed vehicle classes into broader analytical categories\.
Traffic Count\. Number of vehicle crossings associated with the facility, direction, payment method, vehicle class, and hourly observation period\. This serves as the primary traffic volume metric in the dataset\.

In [8]:
# ---------------------------------------------------------------------
# 1.1.3.2 Rename Bridges & Tunnels columns
# ---------------------------------------------------------------------

bridge_tunnel_staged = bridge_tunnel_raw.rename(
    columns={
        "Transit Timestamp": "transit_timestamp",
        "Date": "date",
        "Hour": "hour",
        "Facility ID": "facility_id",
        "Facility": "facility",
        "Direction": "direction",
        "Payment Method": "payment_method",
        "Vehicle Class": "vehicle_class",
        "Vehicle Class Description": "vehicle_class_description",
        "Vehicle Class Category": "vehicle_class_category",
        "Traffic Count": "traffic_count",
    }
)

bridge_tunnel_staged.head()

,transit_timestamp,date,hour,facility_id,facility,direction,payment_method,vehicle_class,vehicle_class_description,vehicle_class_category,traffic_count
0,01/01/2023 12:00:00 AM,01/01/2023,0,27,Queens Midtown Tunnel,Westbound to Manhattan,Tolls by Mail,9,motorcycle,Motorcycle,2
1,01/01/2023 12:00:00 AM,01/01/2023,0,28,Hugh L. Carey Tunnel,Northbound to Manhattan,Tolls by Mail,1,2-axle passenger car,Car,83
2,01/01/2023 12:00:00 AM,01/01/2023,0,29,Throgs Neck Bridge,Southbound to Queens,E-ZPass,6,3-axle truck,Truck,1
3,01/01/2023 12:00:00 AM,01/01/2023,0,28,Hugh L. Carey Tunnel,Northbound to Manhattan,E-ZPass,1,2-axle passenger car,Car,404
4,01/01/2023 12:00:00 AM,01/01/2023,0,30,Verrazzano - Narrows Bridge,Eastbound to Brooklyn,E-ZPass,8,5-axle truck,Truck,3


## 1\.1\.3\.3 Validate core fields and missingness

Now that the Bridges & Tunnels dataset has been loaded and the field names have been standardized, let's perform some basic QA before staging the data for downstream processing\. We'll validate field completeness, temporal coverage, facility coverage, payment methods, vehicle classifications, and traffic count distributions to identify any potential data quality issues early\. Since this dataset is relatively small and well\-structured compared to Bus and TLC, the goal here is primarily to confirm that the source data aligns with expectations and can be safely carried forward into the temporal standardization phase\.

In [9]:
# ---------------------------------------------------------------------
# 1.1.3.3 Validate core fields and missingness
# ---------------------------------------------------------------------

bridge_tunnel_missingness_df = (
    bridge_tunnel_staged
    .isna()
    .sum()
    .reset_index()
    .rename(columns={
        "index": "column",
        0: "missing_count"
    })
)

bridge_tunnel_missingness_df["missing_pct"] = (
    bridge_tunnel_missingness_df["missing_count"] / len(bridge_tunnel_staged)
)

display(bridge_tunnel_missingness_df)

,column,missing_count,missing_pct
0,transit_timestamp,0,0.0
1,date,0,0.0
2,hour,0,0.0
3,facility_id,0,0.0
4,facility,0,0.0
5,direction,0,0.0
6,payment_method,0,0.0
7,vehicle_class,0,0.0
8,vehicle_class_description,0,0.0
9,vehicle_class_category,0,0.0


In [10]:
# ---------------------------------------------------------------------
# Check duplicate records
# ---------------------------------------------------------------------

bridge_tunnel_duplicate_count = bridge_tunnel_staged.duplicated().sum()

print(f"Duplicate records: {bridge_tunnel_duplicate_count:,}")
print(f"Duplicate record share: {bridge_tunnel_duplicate_count / len(bridge_tunnel_staged):.4%}")

Duplicate records: 0
Duplicate record share: 0.0000%


In [11]:
# ---------------------------------------------------------------------
# Validate temporal coverage
# ---------------------------------------------------------------------

bridge_tunnel_temporal_coverage_df = pd.DataFrame({
    "field": ["transit_timestamp", "date", "hour"],
    "min_value": [
        bridge_tunnel_staged["transit_timestamp"].min(),
        bridge_tunnel_staged["date"].min(),
        bridge_tunnel_staged["hour"].min()
    ],
    "max_value": [
        bridge_tunnel_staged["transit_timestamp"].max(),
        bridge_tunnel_staged["date"].max(),
        bridge_tunnel_staged["hour"].max()
    ],
    "distinct_values": [
        bridge_tunnel_staged["transit_timestamp"].nunique(),
        bridge_tunnel_staged["date"].nunique(),
        bridge_tunnel_staged["hour"].nunique()
    ]
})

display(bridge_tunnel_temporal_coverage_df)

,field,min_value,max_value,distinct_values
0,transit_timestamp,01/01/2023 01:00:00 AM,12/31/2025 12:00:00 PM,29468
1,date,01/01/2023,12/31/2025,1228
2,hour,0,23,24


Findings: The Bridges & Tunnels dataset exhibited complete coverage across all fields, with no missing values detected in any of the 11 attributes\. Additionally, no duplicate records were identified among the 5\.9 million observations\. Temporal coverage spans January 1, 2023 through December 31, 2025 and includes all 24 hourly reporting periods\. Overall, the dataset appears complete and internally consistent, with no immediate data quality concerns identified during initial validation\.

## 1\.1\.3\.4 Validate categorical field coverage

Next, let's validate the categorical dimensions of the Bridges & Tunnels dataset\. These fields define how traffic observations are segmented across facilities, travel directions, payment methods, and vehicle classifications\. Understanding the coverage and cardinality of these dimensions helps confirm that all expected categories are represented in the source data and provides an early view into the structure of the crossing observations that will ultimately be incorporated into the multimodal mobility framework\.

In [12]:
# ---------------------------------------------------------------------
# 1.1.3.4 Validate categorical field coverage
# ---------------------------------------------------------------------

bridge_tunnel_categorical_coverage_df = pd.DataFrame({
    "field": [
        "facility_id",
        "facility",
        "direction",
        "payment_method",
        "vehicle_class",
        "vehicle_class_description",
        "vehicle_class_category"
    ],
    "distinct_values": [
        bridge_tunnel_staged["facility_id"].nunique(),
        bridge_tunnel_staged["facility"].nunique(),
        bridge_tunnel_staged["direction"].nunique(),
        bridge_tunnel_staged["payment_method"].nunique(),
        bridge_tunnel_staged["vehicle_class"].nunique(),
        bridge_tunnel_staged["vehicle_class_description"].nunique(),
        bridge_tunnel_staged["vehicle_class_category"].nunique()
    ]
})

display(bridge_tunnel_categorical_coverage_df)

,field,distinct_values
0,facility_id,10
1,facility,10
2,direction,15
3,payment_method,2
4,vehicle_class,28
5,vehicle_class_description,22
6,vehicle_class_category,4


Findings: First, the dataset is much more compact than TLC, Bus, or Subway from a categorical perspective: only 10 facilities, 2 payment methods, 4 vehicle class categories, and 22 detailed vehicle classes\. That's a good sign because it means most of these dimensions can likely be retained without creating a high\-cardinality feature explosion\.

In [13]:
# ---------------------------------------------------------------------
# Facility coverage
# ---------------------------------------------------------------------

bridge_tunnel_facility_counts_df = (
    bridge_tunnel_staged["facility"]
    .value_counts()
    .rename_axis("facility")
    .reset_index(name="observation_count")
)

display(bridge_tunnel_facility_counts_df)

,facility,observation_count
0,Verrazzano - Narrows Bridge,892547
1,Throgs Neck Bridge,840809
2,Bronx - Whitestone Bridge,830043
3,Robert F. Kennedy Bridge Bronx,815146
4,Queens Midtown Tunnel,568441
5,Cross Bay Bridge,489044
6,Hugh L. Carey Tunnel,466405
7,Marine Parkway Bridge,407803
8,Robert F. Kennedy Bridge Manhattan,318763
9,Henry Hudson Bridge,312374


Findings: Second, facility\-level traffic is fairly concentrated\. The Verrazzano\-Narrows Bridge, Throgs Neck Bridge, Bronx\-Whitestone Bridge, and RFK Bronx crossing account for a substantial share of observations\. This suggests that any future facility\-level anomaly detection or congestion\-pricing impact analysis may naturally be driven by a relatively small number of major crossings\.

In [14]:
# ---------------------------------------------------------------------
# Direction coverage
# ---------------------------------------------------------------------

bridge_tunnel_direction_counts_df = (
    bridge_tunnel_staged["direction"]
    .value_counts()
    .rename_axis("direction")
    .reset_index(name="observation_count")
)

display(bridge_tunnel_direction_counts_df)

,direction,observation_count
0,Southbound to Queens,1040787
1,Northbound to Bronx,981941
2,Westbound to Staten Island,448710
3,Eastbound to Brooklyn,443837
4,Northbound to Manhattan or Bronx,412433
5,Southbound to Manhattan or Queens,402713
6,Eastbound to Bronx or Queens,318763
7,Eastbound to Queens,289623
8,Westbound to Manhattan,278818
9,Southbound,249939


In [15]:
# ---------------------------------------------------------------------
# Payment method coverage
# ---------------------------------------------------------------------

bridge_tunnel_payment_method_counts_df = (
    bridge_tunnel_staged["payment_method"]
    .value_counts()
    .rename_axis("payment_method")
    .reset_index(name="observation_count")
)

display(bridge_tunnel_payment_method_counts_df)

,payment_method,observation_count
0,E-ZPass,3332552
1,Tolls by Mail,2608823


Findings: Third, payment method coverage is extremely simple, consisting entirely of E\-ZPass and Tolls by Mail\. This creates an interesting downstream opportunity because payment method may serve as a proxy for commuter behavior, travel frequency, or local versus occasional usage patterns\. Changes in the relative mix of E\-ZPass versus Tolls by Mail before and after congestion pricing may provide an additional signal beyond overall traffic volume\.

In [16]:
# ---------------------------------------------------------------------
# Vehicle class description coverage
# ---------------------------------------------------------------------

bridge_tunnel_vehicle_class_counts_df = (
    bridge_tunnel_staged["vehicle_class_description"]
    .value_counts()
    .rename_axis("vehicle_class_description")
    .reset_index(name="observation_count")
)

display(bridge_tunnel_vehicle_class_counts_df)

,vehicle_class_description,observation_count
0,2-axle passenger car,1110368
1,2-axle truck,1000096
2,3-axle truck,859528
3,motorcycle,611956
4,5-axle truck,603253
5,4-axle truck,556809
6,2-axle passenger car w/trailer,400456
7,2 or 3-axle franchise bus,338379
8,6-axle or greater truck,192641
9,3-axle passenger car,63510


Findings: Finally, vehicle composition appears to be more diverse than expected\. While passenger vehicles dominate, there is substantial representation from commercial truck traffic, buses, motorcycles, and higher\-axle freight vehicles\. This may prove valuable later because congestion pricing impacts may differ across vehicle types\. Retaining vehicle\-class distributions during aggregation could allow future analysis of whether traffic reductions are concentrated among passenger vehicles, freight traffic, or specific commercial vehicle segments\.

## 1\.1\.3\.6 Validate traffic count values

Now let's validate the primary measure in the Bridges & Tunnels dataset: traffic counts\. The goal of this QA step is to confirm that crossing volumes fall within reasonable ranges, contain no unexpected negative values, and exhibit distributions consistent with hourly traffic activity across the MTA bridge and tunnel network\. Since traffic count serves as the primary analytical metric in this dataset, validating its integrity is an important prerequisite before staging the data for downstream temporal standardization and multimodal integration\.

In [17]:
bridge_tunnel_staged["traffic_count"].describe()

count    5.941375e+06
mean     1.905842e+02
std      6.530846e+02
min      1.000000e+00
25%      2.000000e+00
50%      7.000000e+00
75%      3.200000e+01
max      1.424400e+04
Name: traffic_count, dtype: float64

In [18]:
(bridge_tunnel_staged["traffic_count"] <= 0).sum()

0

In [19]:
bridge_tunnel_staged["traffic_count"].quantile(
    [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

0.01       1.0
0.05       1.0
0.25       2.0
0.50       7.0
0.75      32.0
0.95    1240.0
0.99    3462.0
Name: traffic_count, dtype: float64

Findings: Traffic count values exhibited complete coverage with no negative or zero\-volume observations detected\. Hourly crossing volumes ranged from 1 to 14,244 vehicles, with a median value of 7 and a mean of approximately 191 crossings per observation, indicating a strongly right\-skewed distribution\. The 95th and 99th percentiles reached 1,240 and 3,462 crossings respectively, suggesting that a relatively small number of high\-volume facility, direction, and vehicle\-class combinations account for a substantial share of total traffic activity\. This distribution appears consistent with expected transportation patterns and may warrant normalization or transformation techniques during future modeling and anomaly detection workflows\.

## 1\.1\.3\.7 Save staged Bridges & Tunnels parquet

With QA complete, the Bridges & Tunnels dataset can now be staged for downstream processing\. The staged parquet file will serve as the canonical output of the loading phase and provide a compressed, analytics\-friendly representation of the source data for subsequent temporal and spatial standardization workflows\.

In [20]:
# ---------------------------------------------------------------------
# 1.1.3.6 Save staged Bridges & Tunnels parquet
# ---------------------------------------------------------------------

bridge_tunnel_staged.to_parquet(
    BRIDGE_TUNNEL_STAGED_PARQUET_PATH,
    index=False
)

print(
    f"Saved staged Bridges & Tunnels parquet to:\n"
    f"{BRIDGE_TUNNEL_STAGED_PARQUET_PATH}"
)

Saved staged Bridges & Tunnels parquet to:
pipeline_data/1.1.3.bridge_tunnel_hourly_crossings/bridge_tunnel_hourly_crossings_staged.parquet


In [21]:
bridge_tunnel_parquet = pd.read_parquet(
    BRIDGE_TUNNEL_STAGED_PARQUET_PATH
)

print("Original shape:", bridge_tunnel_staged.shape)
print("Parquet shape :", bridge_tunnel_parquet.shape)

Original shape: (5941375, 11)
Parquet shape : (5941375, 11)


## Close

The Bridges and Tunnels dataset was successfully loaded, validated, and staged for downstream processing\. The final dataset contains approximately 5\.9 million observations spanning January 2023 through December 2025 and exhibited complete field coverage with no missing values or duplicate records detected\. Facility, direction, payment method, and vehicle classification dimensions aligned with expectations, while traffic count values exhibited valid ranges and plausible distributions\. The resulting staged parquet file provides a compressed, analytics\-ready representation of bridge and tunnel crossing activity and will serve as the input to the temporal standardization phase, where common date, time, temporal bucket, and congestion\-pricing features will be created to support integration with the project's broader multimodal mobility framework\.


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a322346-8e1e-4650-8cef-fe9b767d96fb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>